# Imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os
from pathlib import Path

from typing import List, Union

import torch #PyTorch

from tqdm.notebook import tqdm

import cv2 #OpenCV

# Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Config
# Path to dataset (Each group member may need to modify this)
DATASET_RAW_FILEPATH_IN_DRIVE = "/content/drive/MyDrive/COMP-SCI 5530 - Principles of Data Science/Project/data_raw/deepfake-and-real-images.zip"

In [ ]:
# Copy data clean zip from Google Drive
!cp "{DATASET_RAW_FILEPATH_IN_DRIVE}" "/content/data_raw.zip"

In [ ]:
# Unzip data raw zip
!unzip "/content/data_raw.zip"

In [ ]:
# Print GPU info
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# Attempt to run on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Running on:", device)

# Helper Fuctions

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def get_faces(img_path: str) -> List[np.ndarray]:
    image = cv2.imread(img_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces_bboxes = face_cascade.detectMultiScale(gray, 1.3, 5)

    face_images = []
    for (x, y, w, h) in faces_bboxes:
        cropped_face = image[y:y+h, x:x+w]
        face_images.append(cropped_face)

    return face_images

# Cleaning Data

In [ ]:
raw_dir_root = "/content/Dataset"
clean_dir_root = "/content/data_clean"

counter = 0

# Get total num files in raw data
total_files = 0
for root, dirs, files in os.walk(raw_dir_root):
  total_files += len(files)

with tqdm(total = total_files, desc="Cleaning Data") as pbar:
  for folder in os.listdir(raw_dir_root):
    folder_path = os.path.join(raw_dir_root, folder)
    for subfolder in os.listdir(folder_path):
      subfolder_path = os.path.join(folder_path, subfolder)
      save_path = f"{clean_dir_root}/{folder}/{subfolder}"
      os.makedirs(save_path, exist_ok=True)
      for img_name in os.listdir(subfolder_path):
        img_path = os.path.join(subfolder_path, img_name)

        faces = get_faces(img_path = img_path)
        for face in faces:
          clean_img_path = os.path.join(save_path, img_name)
          cv2.imwrite(clean_img_path, face)

        pbar.update(1)

In [ ]:
!zip -r "data_clean.zip" "data_clean"

In [ ]:
!cp "/content/data_clean.zip" "/content/drive/MyDrive"

In [ ]:
from google.colab import runtime
runtime.unassign()